### Week 1 

1. In Coding Quiz 1, you are asked to find the distance of the farthest match in a set.  Is this farthest match distance too far to be a meaningful match?

To answer this question it is important to first understand what matching is in terms of causal inference and what methods we are applying to the data. According to the reading, "Matching is the process of closing back doors between a treatment and an outcome by constructing comparison groups that are similar according to a set of matching variables" (Huntington-Klein, 2022). The need to close these back doors stems from confounders (Z) which can affect both X (treatment) and Y (outcome) making it hard to understand the true impact of the treatment (X). In class we learned about the strong correlation between ice cream sales and shark attacks but this relationship would not be meaningful without understanding the confounding impact of the summer heat which was responsible for the increase in both. The temperature in this instance is creating a back door path between shark attacks and ice cream sales causing a bias which would require a balancing of confounders.

Now that we have identified what matching is in terms of casual inference and the goal of closing back doors it is time to select the approach to matching. From the reading we learned there are two main approaches to matching with either distance matching or propensity score matching. The book states, "Distance matching says'observations are similar if they have similar values of the matching variables'" (Huntingon-Klein, 2022). The latter is defined as, "observations are similar if they were equally likely to be treated, 'in other words have equal treatment propensity'"(Huntingon-Klein, 2022). This was the approach used in coding quiz 1 and the farthest match found was at 0.210217. 

Lastly, before we can answer the question of whether or not this match distance is meaningful or not we must decide if we are selecting matches or constructing a matched weighted sample. The former approach of selecting matches gives everyone in the set an equal weight. On the other hand, a constructed weight match sample would give more weight to matches that were better related and less weight to matches that were farther apart. I would argue that the latter is the better approach in this example and in order to understand the weights it would be important to see the distribution of all matching distances. In this dataset about 75% of matches were within 0.056 making the farthest match of 0.210217 an definitive outlier. I would argue that including this match could introduce bias and I would look to remove it in this instance.     

No AI used


2. In Coding Quiz 1, there are two approaches to matching: 
(A) Picking the best match X = 0 corresponding to each X = 1 using Z values.
(B) Using radius_neighbors to pick all matches X = 0 within a distance of 0.2 of each X = 1.

Invent your own type of matching similar to 1 and 2 (or look one up on the internet), which has a different way to pick the matches in X = 0.  Clearly explain the approach you invented or found.


I found the Optimal Matching approach online through the website https://pmc.ncbi.nlm.nih.gov/articles/PMC2943670/#S10 which I would use to pick the matchines in X = 0. From the above, option (A) is the nearest-neighboar approach to matching where we are looking to match the control to the closest treated unit and then repeat the process so on and so forth. However this approach can leave the matches at the end with farther distances because the closer option was already selected earlier on. With the radius matching in option (B) we are may include weak matches with the distance of .2. For example if we had 90% of the matches within .05 and decided to equally weight the other 10% at .199 just because it was within the .2 limit it does not seem like this would be the best approach. 

The optimal matching approach is different from option (A) and (B) in that it is looking for the best overall pair quality for all possible matches simultaneously. This avoids the problem of going one by one in the greedy approach of the nearsest neighbor by fundamentally looking to minimize total distance in matched pairs; this helps avoid an order bias as well. This approach also avoids giving bad matches equal weights like in the example of the radius nearest neighbors. The optimal matching approach does require more computational effort because it is not as simple as the greedy approach in option (A) and the authors state that its not always a better covariate balance.  

No AI used

### Week 2

1. Invent an example situation that would use fixed effects.

When understanding causal inference or causality it its important to understand the correct causal diagram of what to control for, and to properly measure all the variables we need to control for. This is not always possible and very difficult to identify at times which is why fixed effects is so important. According to the textbook, "Fixed effects is a method of controlling for all variables, whether they're observed or not, as long as they stay constant within some larger category...We just control for the larger category, and in doing so we control for everything that is constant within that category" (Huntington-Klein, 2022).

In DX701 I am working on a policy proposal regarding hate speech on social media where I could theoretically use a fixed effects model to measure. I could attempt to measure the amount of hate speech on twitter before and after Elon Musk purchased the platform. If we apply this situation to the equation the variables would look as follows. The B1 would be the estimated change in hate speech after the purchase of the platform by Musk. The i would be the user account of where the posts were coming from. The t would equal the time periods which could be measured in weeks or months. Alpha_i would be the baseline of hate speech and gamma_t would be the time fixed effects. Lastly there would be a random variable or epsilon that the model could not explain.  

No AI used


2. Write a Python program that performs a bootstrap simulation to find the variance in the mean of the Pareto distribution when different samples are taken.  Explain what you had to do for this.  As you make the full sample size bigger (for the same distribution), what happens to the variance of the mean of the samples?  Does it stay about the same, get smaller, or get bigger?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ALPHA = 2.5  # shape
XM = 1.0     # scale
B = 10000    #resamples
N_LIST = [20, 50, 100, 500, 1000]
MASTER_SEED = 7

# RNG 
rng_master = np.random.default_rng(MASTER_SEED)

# Draw Pareto
def pareto(n, rng):
    return XM * (1.0 + rng.pareto(ALPHA, size=n))

# Boot strap 
def manual_bootstrap(sample, B=10000, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    n = len(sample)
    boot_means = np.empty(B, dtype=float)
    for b in range(B):
        idx = rng.integers(0, n, size=n)  
        resample = sample[idx]
        boot_means[b] = resample.mean()
    se = boot_means.std(ddof=1)
    ci_low, ci_high = np.quantile(boot_means, [0.025, 0.975])
    return se, se**2, ci_low, ci_high, boot_means

rows = []
for n in N_LIST:
    # independent RNG 
    rng = np.random.default_rng(rng_master.integers(0, 2**32-1))
    sample = pareto(n, rng)
    se, var, ci_low, ci_high, boot_means = manual_bootstrap(sample, B=B, rng=rng)
    rows.append({
        "n": n,
        "sample_mean": float(sample.mean()),
        "boot_SE_mean": float(se),
        "boot_VAR_mean": float(var),
        "CI_low": float(ci_low),
        "CI_high": float(ci_high),
    })

df = pd.DataFrame(rows)

print("Manual bootstrap variance sample mean from Pareto")
print(df.round(6).to_string(index=False))



Manual bootstrap variance of the sample mean from Pareto
   n  sample_mean  boot_SE_mean  boot_VAR_mean   CI_low  CI_high
  20     1.646095      0.150792       0.022738 1.376105 1.967278
  50     1.789649      0.287183       0.082474 1.419952 2.419530
 100     1.824556      0.306979       0.094236 1.420810 2.527719
 500     1.694070      0.061847       0.003825 1.581569 1.825153
1000     1.676333      0.035557       0.001264 1.609917 1.750036
